In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 根据图片提供的数据构建字典
data = {
    'Element': ['Li', 'Be', 'B', 'Na', 'Mg', 'Al', 'Si', 'S', 'K', 'Ca', 
                'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Zn', 'In', 'Sb', 
                'La', 'Sm', 'Eu', 'Yb', 'Lu', 'Os', 'Ir', 'Au', 'Pb', 'U'],
    'CI_Abundance': [55.59, 0.6116, 18.816, 55300, 1045000, 82730, 1000000, 426600, 3607, 61670, 
                     2574, 276.7, 13340, 9107, 872700, 2270, 48350, 1212, 0.1779, 0.3126, 
                     0.4424, 0.2672, 0.1003, 0.2564, 0.0380, 0.7046, 0.6404, 0.1946, 3.332, 0.00893],
    'Cond_T': [1225, 1490, 964, 970, 1340, 1650, 1340, 648, 1000, 1518, 
               1549, 1455, 1301, 1190, 1337, 1356, 1354, 684, 470, 912, 
               1544, 1560, 1338, 1493, 1598, 1812, 1603, 1284, 520, 1580]
}

# 构建 DataFrame
df_elements = pd.DataFrame(data)

print("基础数据表加载成功！数据维度:", df_elements.shape)

In [ ]:
# 1. 计算全局归一化（原子百分比）
total_abundance = df_elements['CI_Abundance'].sum()
df_elements['CI_norm_100'] = (df_elements['CI_Abundance'] / total_abundance) * 100

# 2. 按冷凝温度从小到大升序排序，并重置索引
df_elements = df_elements.sort_values(by='Cond_T').reset_index(drop=True)

print("按冷凝温度排序后的数据 (前5行)：")
display(df_elements.head()) if 'display' in globals() else print(df_elements.head())

In [ ]:
thresholds = [700, 1200, 1400]

# 遍历设定的温度阈值
for T in thresholds:
    # 筛选大于阈值的记录
    mask = df_elements['Cond_T'] > T
    
    # 计算满足条件的元素的 CI_norm_100 总和
    subset_sum = df_elements.loc[mask, 'CI_norm_100'].sum()
    
    # 生成重新归一化列：满足条件的重新计算百分比，不满足的赋 0
    col_norm = f'Norm_Sub_>{T}'
    df_elements[col_norm] = np.where(mask, (df_elements['CI_norm_100'] / subset_sum) * 100, 0)
    
    # 生成标准化列 (相对于初始 CI 丰度)
    col_std = f'Std_>{T}'
    df_elements[col_std] = df_elements[col_norm] / df_elements['CI_norm_100']

print("处理完成，包含所有归一化与标准化数据的数据表如下：")
display(df_elements) if 'display' in globals() else print(df_elements)

In [ ]:
# 创建宽幅绘图画布
fig, ax = plt.subplots(figsize=(12, 6))
elements = df_elements['Element']

# 绘制三条不同温度阈值的折线（直接使用包含 0 的原始数据）
ax.plot(elements, df_elements['Std_>700'], marker='o', markersize=6, linewidth=2, label='>700 K Component')
ax.plot(elements, df_elements['Std_>1200'], marker='s', markersize=6, linewidth=2, label='>1200 K Component')
ax.plot(elements, df_elements['Std_>1400'], marker='^', markersize=6, linewidth=2, label='>1400 K Component')

# 核心优化：使用 对称对数坐标 (symlog)
# linthresh=0.1 表示在 -0.1 到 0.1 之间使用线性坐标，其余部分使用对数坐标
ax.set_yscale('symlog', linthresh=0.1)

# 添加基准辅助线，增强可读性
ax.axhline(0, color='black', linewidth=1, alpha=0.8) # 0值基准线
ax.axhline(1, color='gray', linestyle='-.', alpha=0.8, label='CI Baseline (y=1)') # 1倍CI基准线

# 绘图设置
ax.set_xlabel('Elements (Sorted by Condensation Temperature)', fontsize=12, fontweight='bold')
ax.set_ylabel('CI Standardized Abundance (Symlog Scale)', fontsize=12, fontweight='bold')
ax.set_title('Standardized Elemental Abundance for Different Temperature Components', fontsize=14)

# 添加图例和网格
ax.legend(fontsize=11, loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.7)

# 强制限制 X 轴范围，确保所有 30 个元素全部显示出来
ax.set_xlim(-1, len(elements))

# 旋转 X 轴标签
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 1. 使用您化简的优雅公式计算混合成分的标准化丰度
df_elements['Mix_Std'] = (
    0.60 
    + 0.385 * df_elements['Std_>1200'].fillna(0) 
    + 0.015 * df_elements['Std_>1400'].fillna(0)
)

# 2. 创建绘图画布
fig, ax = plt.subplots(figsize=(12, 6))

# 3. 绘制混合成分折线
ax.plot(df_elements['Element'], df_elements['Mix_Std'], 
        marker='D', color='purple', markersize=6, linewidth=2, 
        label='Modeled Mixture\n(60% CI + 38.5% >1200K + 1.5% >1400K)')

# 4. 填充区域直观显示亏损与富集 (基准线为 y=1)
ax.fill_between(df_elements['Element'], df_elements['Mix_Std'], 1, 
                where=(df_elements['Mix_Std'] < 1), color='red', alpha=0.1, interpolate=True)
ax.fill_between(df_elements['Element'], df_elements['Mix_Std'], 1, 
                where=(df_elements['Mix_Std'] >= 1), color='blue', alpha=0.1, interpolate=True)

# 5. 添加 CI 球粒陨石基准参考线
ax.axhline(1, color='black', linestyle='-.', alpha=0.5, label='CI Chondrite Baseline (y=1)')

# 6. 绘图细节设置
ax.set_xlabel('Elements (Sorted by Condensation Temperature)', fontsize=12, fontweight='bold')
ax.set_ylabel('CI Standardized Abundance', fontsize=12, fontweight='bold')
ax.set_title('Standardized Elemental Abundance for Modeled Mixture', fontsize=14)

ax.legend(fontsize=11, loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.7)

# 强制限制 X 轴范围，确保元素占满全长
ax.set_xlim(-1, len(df_elements['Element']))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()